In [ ]:
#imports
import pandas as pd
import numpy as np
import os
import joblib 

import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
%matplotlib inline

'''import plotly.offline as py
import plotly.graph_objs as go
import plotly.tools as tls
py.init_notebook_mode(connected=True)'''


print(os.listdir("../data"))
data_path = "../data/pipe_conditions.csv"

import warnings
warnings.filterwarnings('ignore')

['pipe_condition_class_synthetic.csv']


In [ ]:
Seed = 42
np.random.seed(Seed)

In [ ]:
#Load Data
base_path = os.path.dirname(os.getcwd()) 
data_path = os.path.join(base_path, 'data', 'pipe_condition_class_synthetic.csv')

df = pd.read_csv(data_path)
print(df.head())
df.info()

   Age Material  Diameter  Slope  Depth      Length  Soil PH Soil Type  \
0   57      VCP         8   0.34  11.32  221.045734      5.7      Rock   
1   35      VCP        12   0.40   7.00  342.975986      6.8      Clay   
2   28      PVC         8   0.50   8.58  295.072420      5.4      Sand   
3   31      PVC        15   0.17  10.50  492.085601      8.2      Clay   
4   58      PVC         8   0.36   8.58  334.559947      8.2      Clay   

  Road Type  Condition Rating  
0    Street                 3  
1    Street                 5  
2    Street                 1  
3    Street                 1  
4     Alley                 3  
<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Age               30000 non-null  int64  
 1   Material          30000 non-null  str    
 2   Diameter          30000 non-null  int64  
 3   Slope             30000 n

In [12]:
df.describe()

,Age,Diameter,Slope,Depth,Length,Soil PH,Condition Rating
count,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000
mean,47.651267,11.278667,0.779142,8.427552,280.989532,6.870208,2.105267
std,20.719162,9.304710,1.329829,3.160971,194.936076,1.309579,1.199458
min,5.000000,6.000000,0.000000,0.090000,5.764852,4.100000,1.000000
25%,30.000000,8.000000,0.300000,7.000000,160.173861,5.700000,1.000000
50%,51.000000,8.000000,0.400000,8.000000,261.116710,7.500000,2.000000
75%,65.000000,10.000000,0.670000,10.000000,336.869356,8.200000,3.000000
max,121.000000,90.000000,52.897268,77.277406,2245.996774,8.200000,5.000000


In [ ]:
#Analyze the Data (Plots)
#Create Features and Target Variable
x = df.drop('Condition Rating', axis = 1)
y = df['Condition Rating']

,Age,Diameter,Slope,Depth,Length,Soil PH,Condition Rating
count,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000
mean,47.651267,11.278667,0.779142,8.427552,280.989532,6.870208,2.105267
std,20.719162,9.304710,1.329829,3.160971,194.936076,1.309579,1.199458
min,5.000000,6.000000,0.000000,0.090000,5.764852,4.100000,1.000000
25%,30.000000,8.000000,0.300000,7.000000,160.173861,5.700000,1.000000
50%,51.000000,8.000000,0.400000,8.000000,261.116710,7.500000,2.000000
75%,65.000000,10.000000,0.670000,10.000000,336.869356,8.200000,3.000000
max,121.000000,90.000000,52.897268,77.277406,2245.996774,8.200000,5.000000


In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, AdaBoostClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

In [56]:
#Split Categorical and Numerical Features
numerical_cols = ['Age','Diameter','Slope','Depth','Length','Soil PH']
categorical_cols = ['Material','Road Type','Soil Type']

#Training and Testing Datasets
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
print("Training features shape:", X_train.shape)
print("Testing features shape:", X_test.shape)

Training features shape: (24000, 9)
Testing features shape: (6000, 9)


In [57]:
#Imputation & Encoding, Scaling, and Model Pipeline
numerical_pipeline = Pipeline([
    ('imputer',SimpleImputer(strategy= 'mean')),
    ('scaler',StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')), 
    ('encoder',OneHotEncoder(handle_unknown='ignore'))
])

#Join Pipeline
preprocessor = ColumnTransformer([
    ('num',numerical_pipeline,numerical_cols),
    ('cat',categorical_pipeline,categorical_cols)
])

#Model Pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier())
])

param_grid = [
    {
        'model': [RandomForestClassifier(class_weight='balanced', random_state=42)],
        'model__max_depth': [5, 10, 20, None],
        'model__n_estimators': [100]
    },
    {
        'model': [HistGradientBoostingClassifier(class_weight='balanced', random_state=42)],
        'model__max_depth': [5, 10, 20, None]
    }
]

grid_search = GridSearchCV(
    estimator=pipeline, 
    param_grid=param_grid, 
    cv=5, 
    scoring='f1_macro', 
    n_jobs=-1, 
    verbose=2  
)

In [ ]:
#Train and Model
grid_search.fit(X_train, y_train)
prediction = grid_search.predict(X_test)

        
print("Best Model and Parameters:", grid_search.best_params_)
print("Best Cross-Validation Score:", grid_search.best_score_)

#View Encoding
encoded_cols = grid_search.best_estimator_.named_steps['preprocessor'].transformers_[1][1].named_steps['encoder'].get_feature_names_out(categorical_cols)
print("Encoded Categorical Columns:", encoded_cols)

#Evaluate Model
winning_model_name = grid_search.best_estimator_.named_steps['model'].__class__.__name__

print(f"--- Metrics for Best Model: {winning_model_name} ---")
print("Best Hyperparameters:", grid_search.best_params_)

print("\nClassification Report:\n", classification_report(y_test, prediction))
print("Confusion Matrix:\n", confusion_matrix(y_test, prediction))
print("\nAccuracy Score:", accuracy_score(y_test, prediction))

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best Model and Parameters: {'model': RandomForestClassifier(class_weight='balanced', random_state=42), 'model__max_depth': 10, 'model__n_estimators': 100}
Best Cross-Validation Score: 0.2784481093378998
Encoded Categorical Columns: ['Material_PVC' 'Material_RC' 'Material_VCP' 'Road Type_Alley'
 'Road Type_Easement' 'Road Type_Highway' 'Road Type_Street'
 'Soil Type_Clay' 'Soil Type_Gravel' 'Soil Type_Loam' 'Soil Type_Rock'
 'Soil Type_Sand']
--- Metrics for Best Model: RandomForestClassifier ---
Best Hyperparameters: {'model': RandomForestClassifier(class_weight='balanced', random_state=42), 'model__max_depth': 10, 'model__n_estimators': 100}

Classification Report:
               precision    recall  f1-score   support

           1       0.67      0.66      0.67      2958
           2       0.04      0.04      0.04       265
           3       0.47      0.32      0.38      2289
           4       0.07      0.15      0.10    